In [17]:
import pandas as pd
import numpy as np
import glob
import os

In [18]:
# # Solicitará permissão para acessar o seu Google Drive se for executado no Colab
# from google.colab import drive
# drive.mount('/content/drive')

# # Definindo o caminho da nova pasta no Google Drive
# caminho_pasta_base = '/content/drive/MyDrive/Fatec/TCC/dataset'
# caminho_pasta_tratado = '/content/drive/MyDrive/Fatec/TCC/dataset tratado'

In [19]:
# Definindo os caminho da nova pasta no PC pessoal
caminho_pasta_base = '../dataset/cicids2017'
caminho_pasta_tratado = '../dataset tratado/cicids2017'

In [20]:
# Arquivos para o cenário de teste com ataque não visto no treino
nome_dados_treinamento = 'cicids2017_treinamento.csv'
nome_dados_teste = 'cicids2017_teste.csv'

nome_dados_treinamento_sem_portscan = 'cicids2017_treinamento_sem_portscan.csv'
nome_dados_teste_com_portscan = 'cicids2017_teste_com_portscan.csv'

nome_dados_treinamento_sem_XSS = 'cicids2017_treinamento_sem_XSS.csv'
nome_dados_teste_com_XSS = 'cicids2017_teste_com_XSS.csv'

In [21]:
# Mapeia todos os arquivos que terminam com .csv dentro da pasta
arquivos_csv = glob.glob(os.path.join(caminho_pasta_base, "*.csv"))

print(f"Encontrados {len(arquivos_csv)} arquivos para importação.")

def ler_csv_com_encoding(arquivo):
    try:
        return pd.read_csv(arquivo, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(arquivo, encoding="cp1252", low_memory=False)

# Lê cada um dos 8 arquivos e os junta em um único DataFrame
lista_dfs = []
for arquivo in arquivos_csv:
    df_temp = ler_csv_com_encoding(arquivo)
    lista_dfs.append(df_temp)

# Concatena todos os DataFrames da lista empilhando as linhas
df = pd.concat(lista_dfs, ignore_index=True)

# Remove espaços em branco no início/fim dos nomes das colunas
df.columns = df.columns.str.strip()
df_bruto = df.copy()

print("Tamanho total do dataset unificado:", df.shape)
display(df.head())

Encontrados 8 arquivos para importação.
Tamanho total do dataset unificado: (3119345, 85)


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,7/7/2017 3:30,109.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,7/7/2017 3:30,52.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,7/7/2017 3:30,34.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [23]:
# Converte valores infinitos para NaN (nulos)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove todas as linhas que contenham valores nulos
df.dropna(inplace=True)

print("Tamanho do dataset após remoção de nulos/infinitos:", df.shape)

Tamanho do dataset após remoção de nulos/infinitos: (2827876, 85)


In [24]:
from sklearn.preprocessing import MinMaxScaler

# Usa sempre a base original importada, evitando erro ao reexecutar esta célula
df = df_bruto.copy()
df.columns = df.columns.str.strip()

# Separando a variável alvo (Label) dos dados de tráfego
coluna_label = next((col for col in df.columns if col.strip().lower() == 'label'), None)
if coluna_label is None:
    raise KeyError(f"Coluna Label não encontrada. Colunas disponíveis: {list(df.columns)}")

label = df[coluna_label]

# Remove colunas textuais/identificadoras que não devem ser normalizadas
colunas_textuais = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df = df.drop(columns=[coluna_label] + colunas_textuais, errors='ignore')

# Garante que as features restantes sejam numéricas
df = df.apply(pd.to_numeric, errors='coerce')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
linhas_validas = df.notna().all(axis=1)
df = df.loc[linhas_validas].copy()
label = label.loc[linhas_validas].copy()

# Aplicação da escala Min-Max conforme exigido para convergência da Rede Neural
scaler = MinMaxScaler()
df = pd.DataFrame(
    scaler.fit_transform(df), 
    columns=df.columns,
    index=df.index
) 

# Readicionando a coluna 'Label' ao DataFrame tratado
df['Label'] = label.values

print("Dados numéricos normalizados com sucesso.")
display(df.head())

Dados numéricos normalizados com sucesso.


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.006760,0.837186,0.352941,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.001221,0.840070,0.352941,1.016667e-06,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.001221,0.840085,0.352941,5.416666e-07,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.006760,0.705516,0.352941,3.916666e-07,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.006760,0.837156,0.352941,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [25]:
from sklearn.model_selection import train_test_split

df_treino, df_teste = train_test_split(df, test_size=0.3, random_state=42)

print(f"Dados de Treino: {df_treino.shape}")
print(f"Dados de Teste: {df_teste.shape}")

# O comando makedirs cria a pasta "dataset filtrado" caso ela ainda não exista no seu Drive
os.makedirs(caminho_pasta_tratado, exist_ok=True)

# 3. Caminho completo do arquivo CSV que será gerado
caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste)

# 4. Exportando o DataFrame para o formato CSV
df_treino.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: '{caminho_arquivo_treinamento}'")

df_teste.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: '{caminho_arquivo_teste}'")

Dados de Treino: (1979513, 81)
Dados de Teste: (848363, 81)
Dataset de treinamento salvo com sucesso em: '../dataset tratado/cicids2017\cicids2017_treinamento.csv'
Dataset de teste salvo com sucesso em: '../dataset tratado/cicids2017\cicids2017_teste.csv'


In [26]:
# Colocando PortScan somente no teste para simular o cenário de ataque não visto no treino
df_treino_sem_portscan, df_teste_com_portscan = train_test_split(df, test_size=0.3, random_state=42)

# Move todos os registros PortScan do treino para o teste
eh_portscan_treino = df_treino_sem_portscan['Label'].astype(str).str.contains('portscan', case=False, na=False)
portscan_do_treino = df_treino_sem_portscan[eh_portscan_treino].copy()
df_treino_sem_portscan = df_treino_sem_portscan[~eh_portscan_treino].copy()
df_teste_com_portscan = pd.concat([df_teste_com_portscan, portscan_do_treino], ignore_index=True)

print(f"PortScan movido do treino para o teste: {len(portscan_do_treino)}")
print(f"PortScan no treino após mover: {df_treino_sem_portscan['Label'].astype(str).str.contains('portscan', case=False, na=False).sum()}")
print(f"PortScan no teste após mover: {df_teste_com_portscan['Label'].astype(str).str.contains('portscan', case=False, na=False).sum()}")

print(f"Dados de Treino: {df_treino_sem_portscan.shape}")
print(f"Dados de Teste: {df_teste_com_portscan.shape}")

os.makedirs(caminho_pasta_tratado, exist_ok=True)

caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento_sem_portscan)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste_com_portscan)

df_treino_sem_portscan.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: {caminho_arquivo_treinamento}")

df_teste_com_portscan.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: {caminho_arquivo_teste}")

PortScan movido do treino para o teste: 110968
PortScan no treino após mover: 0
PortScan no teste após mover: 158804
Dados de Treino: (1868545, 81)
Dados de Teste: (959331, 81)
Dataset de treinamento salvo com sucesso em: ../dataset tratado/cicids2017\cicids2017_treinamento_sem_portscan.csv
Dataset de teste salvo com sucesso em: ../dataset tratado/cicids2017\cicids2017_teste_com_portscan.csv


In [27]:
# Colocando XSS somente no teste para simular o cenário de ataque não visto no treino
df_treino_sem_xss, df_teste_com_xss = train_test_split(df, test_size=0.3, random_state=42)

# Move todos os registros XSS do treino para o teste
eh_xss_treino = df_treino_sem_xss['Label'].astype(str).str.contains('XSS', case=False, na=False)
xss_do_treino = df_treino_sem_xss[eh_xss_treino].copy()
df_treino_sem_xss = df_treino_sem_xss[~eh_xss_treino].copy()
df_teste_com_xss = pd.concat([df_teste_com_xss, xss_do_treino], ignore_index=True)

print(f"XSS movido do treino para o teste: {len(xss_do_treino)}")
print(f"XSS no treino após mover: {df_treino_sem_xss['Label'].astype(str).str.contains('XSS', case=False, na=False).sum()}")
print(f"XSS no teste após mover: {df_teste_com_xss['Label'].astype(str).str.contains('XSS', case=False, na=False).sum()}")

print(f"Dados de Treino: {df_treino_sem_xss.shape}")
print(f"Dados de Teste: {df_teste_com_xss.shape}")

os.makedirs(caminho_pasta_tratado, exist_ok=True)

caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento_sem_XSS)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste_com_XSS)

df_treino_sem_xss.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: {caminho_arquivo_treinamento}")

df_teste_com_xss.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: {caminho_arquivo_teste}")

XSS movido do treino para o teste: 459
XSS no treino após mover: 0
XSS no teste após mover: 652
Dados de Treino: (1979054, 81)
Dados de Teste: (848822, 81)
Dataset de treinamento salvo com sucesso em: ../dataset tratado/cicids2017\cicids2017_treinamento_sem_XSS.csv
Dataset de teste salvo com sucesso em: ../dataset tratado/cicids2017\cicids2017_teste_com_XSS.csv
